In [3]:
import os
import sys
import logging
import datetime
import numpy as np
import cv2
from deepface import DeepFace


In [5]:
DATASET_PATH = r"C:\Users\hp\OneDrive\Desktop\CESI_UL Internship\Dataset"
OUTPUT_PATH  = r"C:\Users\hp\StressDetection\DL\outputttt"


# Your assigned participants
PARTICIPANTS = [55, 56]


TASKS = ["T1", "T2", "T3"]

In [6]:
WINDOW_SIZE          = 140
STEP                 = 35
DEEPFACE_SAMPLE_RATE = 17    # run DeepFace every 17 frames (~2x per second at 35fps)

In [7]:

os.makedirs(OUTPUT_PATH, exist_ok=True)

log_path = os.path.join(OUTPUT_PATH, "extract_val_log.txt")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger()

In [8]:

log.info("=" * 60)
log.info("extract_val.py — START")
log.info(f"Date/Time     : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log.info(f"Python        : {sys.version}")
log.info(f"NumPy         : {np.__version__}")
log.info(f"OpenCV        : {cv2.__version__}")
try:
    import deepface as df_module
    log.info(f"DeepFace      : {df_module.__version__}")
except Exception:
    log.info("DeepFace      : version unknown")
log.info(f"Dataset path  : {DATASET_PATH}")
log.info(f"Output path   : {OUTPUT_PATH}")
log.info(f"Participants  : {PARTICIPANTS}")
log.info(f"DeepFace rate : every {DEEPFACE_SAMPLE_RATE} frames")
log.info("=" * 60)

2026-06-15 17:45:51,519 — INFO — ============================================================
2026-06-15 17:45:51,519 — INFO — extract_val.py — START
2026-06-15 17:45:51,525 — INFO — Date/Time     : 2026-06-15 17:45:51
2026-06-15 17:45:51,528 — INFO — Python        : 3.9.25 (main, Nov  3 2025, 22:44:01) [MSC v.1929 64 bit (AMD64)]
2026-06-15 17:45:51,528 — INFO — NumPy         : 2.0.2
2026-06-15 17:45:51,528 — INFO — OpenCV        : 4.13.0
2026-06-15 17:45:51,538 — INFO — DeepFace      : 0.0.100
2026-06-15 17:45:51,542 — INFO — Dataset path  : C:\Users\hp\OneDrive\Desktop\CESI_UL Internship\Dataset
2026-06-15 17:45:51,544 — INFO — Output path   : C:\Users\hp\StressDetection\DL\outputttt
2026-06-15 17:45:51,549 — INFO — Participants  : [55, 56]
2026-06-15 17:45:51,552 — INFO — DeepFace rate : every 17 frames
2026-06-15 17:45:51,556 — INFO — ============================================================


In [9]:

def compute_valence(emotion_dict):
    """
    Convert DeepFace emotion probabilities to a single valence score.

    Formula (Stream 2 design — confirm with supervisors):
        positive = happy + 0.5 * surprise
        negative = angry + disgust + fear + sad
        valence  = (positive - negative) / total
        range    : -1.0 to +1.0

    Returns float valence value.
    """
    happy    = emotion_dict.get("happy",    0.0)
    sad      = emotion_dict.get("sad",      0.0)
    angry    = emotion_dict.get("angry",    0.0)
    disgust  = emotion_dict.get("disgust",  0.0)
    fear     = emotion_dict.get("fear",     0.0)
    surprise = emotion_dict.get("surprise", 0.0)
    neutral  = emotion_dict.get("neutral",  0.0)

    positive = happy + 0.5 * surprise
    negative = angry + disgust + fear + sad
    total    = happy + sad + angry + disgust + fear + surprise + neutral

    if total > 0:
        return float((positive - negative) / total)
    return 0.0


def slice_into_windows(buffer, window_size, step):
    """
    Slice a flat buffer into overlapping windows.
    Leftover frames that don't fill a complete window are discarded.
    Returns list of windows, each a list of length window_size.
    """
    windows = []
    total   = len(buffer)
    for start in range(0, total - window_size + 1, step):
        windows.append(buffer[start : start + window_size])
    return windows


def sanity_check(path, label):
    """
    Reload a saved valence .npy file and verify shape and NaN absence.
    Expected shape: (num_windows, 140, 1)
    """
    try:
        arr = np.load(path)
        assert arr.shape[0] > 0,   "zero windows"
        assert arr.shape[1] == 140, f"shape[1] expected 140, got {arr.shape[1]}"
        assert arr.shape[2] == 1,   f"shape[2] expected 1, got {arr.shape[2]}"
        assert not np.isnan(arr).any(), "NaN values found"
        v_min = float(arr.min())
        v_max = float(arr.max())
        log.info(f"  Sanity OK  [{label}] shape={arr.shape} "
                 f"valence_range=[{v_min:.3f}, {v_max:.3f}]")
        return True
    except Exception as e:
        log.warning(f"  Sanity FAIL [{label}] {e}")
        return False


In [10]:

for participant_id in PARTICIPANTS:

    p_str = str(participant_id)
    p_out = os.path.join(OUTPUT_PATH, f"p{p_str}")
    os.makedirs(p_out, exist_ok=True)

    # ── Crash recovery ──────────────────────────────────────────────────────
    all_files = [
        os.path.join(p_out, f"p{p_str}_T{t}_valence.npy")
        for t in ["1", "2", "3"]
    ]
    if all(os.path.exists(f) for f in all_files):
        log.info(f"Participant {p_str} — all 3 valence files exist, skipping")
        continue

    log.info(f"\nParticipant {p_str} — starting valence extraction")

    # ── Task loop ────────────────────────────────────────────────────────────
    for task in TASKS:

        task_num = task[1]  # "1", "2", "3"

        valence_path = os.path.join(p_out, f"p{p_str}_T{task_num}_valence.npy")

        # Individual task crash recovery
        if os.path.exists(valence_path):
            log.info(f"  {task} — file exists, skipping")
            continue

        # ── Open video ───────────────────────────────────────────────────────
        video_path = os.path.join(
            DATASET_PATH,
            f"Participant{participant_id}",
            f"vid_s{participant_id}_{task}.avi"
        )

        if not os.path.exists(video_path):
            log.warning(f"  {task} — video not found: {video_path}")
            continue

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            log.warning(f"  {task} — could not open video: {video_path}")
            continue

        # ── Initialize buffers ───────────────────────────────────────────────
        valence_buffer = []
        last_valence   = 0.0    # neutral starting point
        failed_runs    = 0
        total_frames   = 0
        frame_count    = 0
        deepface_runs  = 0

        # ── PART 1 — Frame loop ──────────────────────────────────────────────
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            total_frames += 1

            if frame_count % DEEPFACE_SAMPLE_RATE == 0:
                deepface_runs += 1
                try:
                    result = DeepFace.analyze(
                        frame,
                        actions=["emotion"],
                        enforce_detection=False,
                        silent=True
                    )
                    # DeepFace returns a list — take first face
                    if isinstance(result, list):
                        result = result[0]

                    emotions     = result["emotion"]
                    last_valence = compute_valence(emotions)

                except Exception as e:
                    failed_runs += 1
                    # keep last_valence unchanged — forward fill
                    log.warning(f"  Frame {total_frames} DeepFace error: {e}")

            # Forward fill for all non-sampled frames
            valence_buffer.append(last_valence)
            frame_count += 1

        cap.release()

        log.info(f"  {task} — frames={total_frames}, "
                 f"deepface_runs={deepface_runs}, "
                 f"failed_runs={failed_runs}")

        # ── PART 2 — Windowing ────────────────────────────────────────────────
        valence_windows = slice_into_windows(valence_buffer, WINDOW_SIZE, STEP)
        num_windows     = len(valence_windows)

        log.info(f"  {task} — windows produced: {num_windows}")

        if num_windows == 0:
            log.warning(f"  {task} — zero windows produced, skipping")
            continue

        # Reshape each window to (140, 1) then stack
        valence_array = np.array(valence_windows, dtype=np.float32)  # (N, 140)
        valence_array = valence_array[:, :, np.newaxis]               # (N, 140, 1)

        # ── PART 3 — Save ─────────────────────────────────────────────────────
        np.save(valence_path, valence_array)
        log.info(f"  {task} — saved valence {valence_array.shape} → {valence_path}")

        # ── PART 4 — Sanity check ─────────────────────────────────────────────
        sanity_check(valence_path, "valence")

log.info("\n" + "=" * 60)
log.info("extract_val.py — COMPLETE")
log.info("=" * 60)

2026-06-15 17:46:22,881 — INFO — 
Participant 55 — starting valence extraction
2026-06-15 17:54:23,091 — INFO —   T1 — frames=6325, deepface_runs=373, failed_runs=0
2026-06-15 17:54:23,095 — INFO —   T1 — windows produced: 177
2026-06-15 17:54:23,103 — INFO —   T1 — saved valence (177, 140, 1) → C:\Users\hp\StressDetection\DL\outputttt\p55\p55_T1_valence.npy
2026-06-15 17:54:23,140 — INFO —   Sanity OK  [valence] shape=(177, 140, 1) valence_range=[-1.000, 1.000]
2026-06-15 18:02:27,497 — INFO —   T2 — frames=6325, deepface_runs=373, failed_runs=0
2026-06-15 18:02:27,500 — INFO —   T2 — windows produced: 177
2026-06-15 18:02:27,508 — INFO —   T2 — saved valence (177, 140, 1) → C:\Users\hp\StressDetection\DL\outputttt\p55\p55_T2_valence.npy
2026-06-15 18:02:27,613 — INFO —   Sanity OK  [valence] shape=(177, 140, 1) valence_range=[-1.000, 1.000]
2026-06-15 18:08:04,583 — INFO —   T3 — frames=6325, deepface_runs=373, failed_runs=0
2026-06-15 18:08:04,583 — INFO —   T3 — windows produced: 1